# 03 · Unlearning: calibration, pilot, main runs and ABA runs
### Machine Unlearning is Extinction, Not Forgetting

This notebook produces every unlearned model, in four stages. Each stage can be switched on or off.

| Stage | What | Jobs | Time on T4 x2 |
|---|---|---|---|
| **A** calibration | Hyper-parameter grid for each method on the **pilot class** (seed 0); ranks the settings | ~33 | ~0.5–1 h |
| **B** pilot | Unlearns the pilot class with every method, for the go/no-go pilot in notebook 04 | 5 | ~5 min |
| **C** main | 10 classes × 3 seeds × 5 methods, each tuned until **matched** (see below) | 150 | ~2–4 h |
| **D** ABA | Same, but unlearning happens in **context B** (gray images), for 5 classes | 75 | ~1–2 h |
| **E** multi-context | Unlearning with every image in a random context (clean / gray / blur), for 5 classes: the extinction-theory remedy | 75 | ~1–2 h |

### Why calibrate on a separate pilot class?
If we tuned hyper-parameters on the classes we later analyse, we could (even unintentionally) pick settings
that make the effect look larger. Tuning on `lamp` and then *freezing* the ranked settings avoids that.

### The matching rule
Different methods forget by different amounts, so comparing them at arbitrary settings would be unfair.
Every model is therefore pushed to the same output-level target: **forget accuracy ≤ 1 % and retain
accuracy within 3 points of the retrained model**. At that point output metrics say forgetting is as good
as retraining. The later tests then check whether the knowledge is really gone.

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| **CIFAR-100 Python** dataset ([fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)) | **Strongly recommended.** Without it CIFAR-100 is downloaded from cs.toronto.edu at ~50 kB/s (56 min in run #1). Check that the setup cell prints `CIFAR-100: found at …`. |
| Output of **notebook 01** (`ext-01-originals`) | The original models (the starting point of every unlearning run) and `experiment_config.json`. |
| Output(s) of **notebook 02** (`ext-02-retrain`, or `ext-02-retrain-a` + `ext-02-retrain-b` if split) | The retrained models' metrics (for the matching rule) and checkpoints (for the gray-context reference in stage D). |
| *(when resuming)* this notebook's own previous output (same name as below) | Finished jobs are skipped and carried over into the new output. |

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

| Parameter | Value | What it does and why this value |
|---|---|---|
| `smoke` | `False` | `True` = a tiny dry run (30 training and 10 test images per class, 1 epoch, width 16) that finishes in minutes. Its files get a `_smoke` suffix, so they never mix with real results. Use it once to test the whole chain, then set `False`. **Must be the same in every notebook.** |
| `seeds` | `[0, 1, 2]` | Independent repetitions (different weight initialisation and data order). Three is the minimum for estimating variance. One statistical unit = (method, class, seed). |
| `forget_classes` | 10 classes | One class from each of 10 *different* CIFAR-100 superclasses, so no single superclass drives the result. The other 4 classes in each superclass are its "siblings" (semantic neighbours). |
| `norm` | `"bn"` | Normalization layer. `"bn"` = BatchNorm, the standard ResNet-18 of the unlearning literature (main experiments). `"gn"` = GroupNorm, a **control arm without running statistics**. The "BatchNorm illusion" (Kalani et al. 2026) showed that recomputing BatchNorm statistics alone can undo apparent forgetting. If the extinction signs also appear with GroupNorm, they are not a BatchNorm artifact. GroupNorm files get a `_gn` suffix, so both arms can live side by side. **Must be the same in every notebook of one arm.** |
| `pilot_class` | `"lamp"` | Used only for hyper-parameter calibration and the go/no-go pilot, then **excluded** from the main analysis. That way hyper-parameters are never tuned on the data that is later tested. |
| `width` | 64 | Channel width of ResNet-18 (64 = the standard model, 11.2 M parameters). |
| `epochs` | 40 | With the OneCycle schedule this reaches about 74–77 % test accuracy on CIFAR-100. The classic 200-epoch recipe gains only ~2–3 points for 5× the compute, which Kaggle's quota can't afford across ~34 models. |
| `batch_size` | 256 | Keeps a T4 busy with mixed precision. The learning rate below is tuned for this size. |
| `lr` | 0.1 | Peak learning rate of the OneCycle schedule. Standard for SGD on CIFAR at batch 256. |
| `pct_start` | 0.25 | Share of training spent warming the learning rate *up* before annealing it down. It stabilises the first epochs. |
| `momentum`, `nesterov` | 0.9, `True` | Standard SGD settings for ResNets. Nesterov momentum converges slightly faster. |
| `weight_decay` | 5e-4 | Standard L2 regularisation for ResNet-18 on CIFAR. |
| `label_smoothing` | 0.0 | Smoothing changes how confident the model is, which would distort the forgetting and recovery measurements. Plain cross-entropy keeps the models comparable with the literature. |
| `amp` | `True` | Mixed precision (fp16) uses the T4's tensor cores: about 2× faster and half the memory, with the same accuracy. |
| `channels_last` | `True` | A memory layout that makes convolutions faster on tensor-core GPUs. |
| `cudnn_benchmark` | `True` | cuDNN tries several convolution algorithms once and keeps the fastest (our image size never changes). It costs bit-exact reproducibility, which is acceptable because we repeat over 3 seeds. |
| `eval_batch_size` | 1024 | Evaluation needs no gradients, so bigger batches are safe and faster. |
| `ckpt_fp16` | `True` | Checkpoints are stored in half precision (≈22 MB instead of 45 MB) to stay well below Kaggle's 20 GB output limit. Every metric is computed *after* the same rounding, so reported numbers match the saved files exactly. |
| `retain_eval_subset` | 5000 | Number of training images used to estimate retain-*train* accuracy. All 49,500 would triple evaluation time for no real gain. |
| `methods` | 5 methods | **FT**: fine-tune on retain data only (also our "natural forgetting" arm). **NegGrad+**: gradient *ascent* on the forget set plus descent on the retain set. **RL** (Random-Label): train forget images with random wrong labels. **SCRUB**: a student moves away from the original (teacher) on forget data and stays close on retain data. **SalUn**: Random-Label updates restricted to the weights most salient for the forget class. Together they cover the main families of approximate unlearning. |
| `forget_batch_size` | 64 | Forget images per step for methods that use them. 500 forget images ≈ 8 batches per pass. |
| `clip_grad` | 5.0 | Gradient-norm clipping for the loss-*maximising* parts (NegGrad+, SCRUB's max step), which can otherwise blow up. |
| `match_forget_max` | 0.01 | A model is **matched** only if its forget-class test accuracy is ≤ 1 % (the retrained model has 0 %). |
| `match_retain_gap` | 0.03 | …and its retain test accuracy is within 3 points of the retrained model. Only matched models pass output-level forgetting, so any later recovery is *hidden* knowledge. |
| `max_tries` | 3 | For each model, try up to the 3 best calibrated settings until one is matched. Unmatched models are kept but flagged, so you can report them. |
| `calib_grid` | `None` (built-in grid) | The hyper-parameter grid for calibration, shown in the table below. `None` uses the default. |
| `calib_limit` | `None` | Testing only: use just the first *n* grid entries per method. |
| `aba_classes` | 5 classes | Classes that also get **ABA-renewal** unlearning: unlearning done in context B (gray images), later tested back in context A (colour). Five classes keep the extra cost at 75 runs. |
| `aba_context` | `"gray"` | Context B. Grayscale changes the visual "context" strongly but keeps object shape, so a model can still recognise the classes. |
| `mctx_classes` | 5 classes | Classes that also get **multi-context unlearning** (stage E, 75 runs). In animals, extinction carried out in *several* contexts relapses less than extinction in one context. This tests whether the same "remedy" works for unlearning. |
| `mctx_contexts` | `["clean", "gray", "blur"]` | Contexts mixed during multi-context unlearning (a random one per image). `clean` is included so normal images are unlearned too. All other contexts (contrast, noise, every CIFAR-100-C corruption) stay **held out** for the renewal test. |
| `time_budget_h` | 11.0 | Kaggle stops sessions at 12 h. The runner stops *starting* new jobs after ~11 h (it estimates each job's length), so the notebook finishes cleanly and its output is saved. Re-run to continue. |
| `n_workers` | `None` | `None` = one worker process per visible GPU (2 on T4 x2, 1 on P100). Set 1 to force a single worker, e.g. to see tracebacks directly. |
| `reuse_ckpt` | `True` | If a job's checkpoint already exists (e.g. saved just before a crash), load it instead of recomputing it. |
| `raise_on_error` | `True` | After all jobs have run, the cell raises an error if any job failed, so failures can't go unnoticed. Finished jobs are always kept. |
| `poll_seconds` | 60 | How often the progress line (`done/total`, elapsed minutes, last log line of each worker) is printed. |
| `RUN_A` … `RUN_E` | `True` | Switch the stages on or off. On a resumed run, finished stages are skipped automatically anyway. |
| `MANUAL_CANDIDATES` | `None` | Advanced: a dict such as `{"FT": [{"lr": 0.02, "epochs": 10}], ...}` that replaces the calibrated settings. Leave it `None` to use calibration. |

The training settings (`width`, `epochs`, …) must still be listed here: they define the model architecture
that is loaded, and they are checked against notebook 01.


**Default calibration grid** (run on the pilot class `lamp`, seed 0):

| Method | Grid | Meaning of the hyper-parameters |
|---|---|---|
| FT | lr ∈ {0.01, 0.02, 0.05} × epochs ∈ {5, 10} | fine-tuning learning rate and passes over the retain set (cosine decay) |
| NegGrad+ | lr ∈ {0.005, 0.01, 0.02} × α ∈ {0.9, 0.95, 0.99}, 5 epochs | α weights retain descent against forget ascent: loss = α·CE_retain − (1−α)·CE_forget |
| RL | lr ∈ {0.005, 0.01, 0.02} × epochs ∈ {3, 5} | learning rate and passes (each retain batch is paired with a batch of randomly relabelled forget images) |
| SCRUB | lr ∈ {5e-4, 1e-3, 5e-3} × max-steps ∈ {3, 5}; epochs = max-steps + 3; α = 0.001, γ = 0.99, T = 4 | max-steps = epochs that include the "move away on forget" phase; γ, α weight CE and distillation on retain; T = distillation temperature (the original paper's defaults) |
| SalUn | lr ∈ {0.005, 0.01, 0.02} × epochs ∈ {3, 5}, mask ratio 0.5 | only the 50 % of weights with the largest forget-loss gradient are updated |

In [ ]:
CFG = U.make_cfg(
    smoke=False,
    seeds=[0, 1, 2],
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],
    pilot_class="lamp",
    norm="bn",                      # "gn" = GroupNorm control arm (see notebook 01)
    width=64, epochs=40, batch_size=256, lr=0.1, pct_start=0.25,
    momentum=0.9, nesterov=True, weight_decay=5e-4, label_smoothing=0.0,
    amp=True, channels_last=True, cudnn_benchmark=True, eval_batch_size=1024,
    ckpt_fp16=True, retain_eval_subset=5000,
    # unlearning
    methods=["FT", "NegGrad+", "RL", "SCRUB", "SalUn"],
    forget_batch_size=64, clip_grad=5.0,
    match_forget_max=0.01, match_retain_gap=0.03, max_tries=3,
    calib_grid=None, calib_limit=None,
    aba_classes=["maple_tree", "tiger", "rose", "shark", "bus"], aba_context="gray",
    mctx_classes=["maple_tree", "tiger", "rose", "shark", "bus"], mctx_contexts=["clean", "gray", "blur"],
    # infrastructure
    time_budget_h=11.0, n_workers=None, reuse_ckpt=True, raise_on_error=True, poll_seconds=60,
)
RUN_A, RUN_B, RUN_C, RUN_D, RUN_E = True, True, True, True, True
MANUAL_CANDIDATES = None
U.check_experiment_config(CFG)

## Stage A: calibration

Every grid setting is applied to the original model (seed 0) for the pilot class. The resulting model is
then compared with the retrained pilot model. `U.select_candidates` ranks the settings of each method:

1. **matched** settings first, the best retain accuracy on top,
2. then unmatched settings, the lowest forget accuracy on top.

The top `max_tries` settings per method are saved to `results/calibration.json`. Stages B–D try them in
that order.

In [ ]:
if RUN_A:
    jobs = U.jobs_calibrate(CFG)
    print(f"{len(jobs)} calibration jobs")
    U.launch("calibrate", jobs, CFG)
    candidates, calib_table = U.select_candidates(CFG)
    display(calib_table.round(4))

**How to read this table.** For each method, at least one row should have `matched = True`. If a method
has *no* matched setting (FT often forgets too weakly), it is still run with its best setting and flagged
`matched = False`. Report it as "could not reach retraining-level forgetting". The main analysis uses
matched models only.

If a method is far from the targets, you can widen its grid via `calib_grid` (e.g. larger learning rates)
and re-run stage A. Grid entries that already ran are skipped.

In [ ]:
candidates = MANUAL_CANDIDATES or U.load_candidates(CFG)
for m, c in candidates.items():
    print(f"{m:9s} tries in order: {c}")

## Stage B: pilot unlearning

Unlearns the pilot class with every method (seed 0). Notebook 04 first runs all extinction tests on these
models only, which takes a few minutes, to decide **go / no-go** before the expensive main run.

In [ ]:
if RUN_B:
    U.launch("unlearn", U.jobs_unlearn(CFG, candidates, kind="unl", pilot=True), CFG)

## Stage C: main unlearning (150 models)

For each (method, class, seed): load the original model, unlearn with the first candidate setting, check
the matching rule, and if it isn't matched try the next candidate (up to `max_tries`). The best model is
saved as `ckpt/unl_<method>_<class>_s<seed>.pt`.

Result columns worth knowing:
- `matched`: passed the matching rule,
- `tries`: how many candidate settings were needed,
- `hp`: the setting that was used,
- `neighbour_test_acc` vs `ref_neighbour_test_acc`: collateral damage on sibling classes compared with the
  retrained model (your co-author's collateral-forgetting question, for free).

In [ ]:
if RUN_C:
    U.launch("unlearn", U.jobs_unlearn(CFG, candidates, kind="unl"), CFG)

## Stage D: ABA unlearning (context B = gray)

**ABA renewal** is the classic extinction design: learn in context A, extinguish in context B, test back in
context A. Here:
- **A** = the original training on colour images,
- **B** = unlearning performed on *grayscale* versions of every image (forget and retain),
- the **test** compares forget-class accuracy in B (gray) and back in A (colour).

For this model the matching rule is checked **in context B**, because that is where it was "extinguished".
Recovery in colour images would then be renewal.

In [ ]:
if RUN_D:
    U.launch("unlearn", U.jobs_unlearn(CFG, candidates, kind="aba"), CFG)

## Stage E: multi-context unlearning (the remedy extinction theory predicts)

Extinction research shows that extinguishing a response in **several contexts** makes it relapse less
(e.g. Gunther, Denniston & Miller, 1998). If unlearning really behaves like extinction, the same should hold:
- during unlearning, every forget and retain image is shown in a **random context** from `mctx_contexts`
  (clean, gray, blur);
- the matching rule is checked on clean images, as for the main models;
- notebook 04 runs all four extinction tests on these models, and notebook 06 compares them with the normal
  models of the same method, class and seed. Renewal is measured only on **held-out** contexts (contrast,
  noise and all CIFAR-100-C corruptions), which the model never saw during unlearning.

**Prediction (H6):** less renewal and less spontaneous recovery than standard unlearning. This turns the
paper from "we observed extinction-like relapse" into "the theory predicted both the failure and a fix".

In [ ]:
if RUN_E:
    U.launch("unlearn", U.jobs_unlearn(CFG, candidates, kind="mctx"), CFG)

## Summary: did the methods reach the matching rule?

`matched_rate` is the share of the 30 (class, seed) models of each method that met the rule. The accuracy
columns are means. `G_related` and `G_unrelated` are the **retraining gaps**: the retrained model's
accuracy minus the unlearned model's, on sibling and on unrelated classes. A larger `G_related` means
unlearning damaged semantic neighbours more than true retraining would.

In [ ]:
import pandas as pd
df = U.load_df("unlearn", CFG)
if df.empty:
    print("No unlearning results yet.")
else:
    main = df[(df["kind"] == "unl") & (df["cls"] != CFG["pilot_class"])].copy()
    main["G_related"] = main["ref_neighbour_test_acc"] - main["neighbour_test_acc"]
    main["G_unrelated"] = main["ref_unrelated_test_acc"] - main["unrelated_test_acc"]
    summary = main.groupby("method").agg(
        n=("matched", "size"), matched_rate=("matched", "mean"),
        forget_test_acc=("forget_test_acc", "mean"), retain_test_acc=("retain_test_acc", "mean"),
        ref_retain_test_acc=("ref_retain_test_acc", "mean"), G_related=("G_related", "mean"),
        G_unrelated=("G_unrelated", "mean"), mean_tries=("tries", "mean"))
    display(summary.round(4))
    aba = df[df["kind"] == "aba"]
    if len(aba):
        print("ABA models: forget accuracy in context B (gray) vs. back in colour (A):")
        display(aba.groupby("method")[["forget_test_acc", "clean_forget_test_acc", "matched"]].mean().round(4))
    mc = df[df["kind"] == "mctx"]
    if len(mc):
        print("Multi-context models: matched rate and accuracies (clean images):")
        display(mc.groupby("method")[["matched", "forget_test_acc", "retain_test_acc", "ref_retain_test_acc"]].mean().round(4))

## Next step

Save a version. Then open **notebook 04** and attach the outputs of notebooks **01, 02 and 03**.

## Name of this run's output

Title this notebook **`ext-03-unlearning`**. When the run has finished, create a dataset from its output with the
same name: open the notebook's viewer page → **Output** → **New Dataset** → title **`ext-03-unlearning`**, Private.
(Or skip the dataset and attach the notebook output directly: *Add Input → Your Work → Notebooks*.)

| Output name | Contains | Attached by |
|---|---|---|
| `ext-03-unlearning` | `calibration.json`, `calibrate` + `unlearn` results, 150 unlearned + 75 ABA + 75 multi-context checkpoints (≈ 7 GB) | notebooks 04, 05, 06 |

- **Re-runs and resumes:** don't create a new name. Add a **New Version** of the same dataset. Each output
  already contains everything from earlier runs (it's carried over), so only the latest version matters.
  In the notebooks that use it, update the input to the new version.
- **Smoke (dry) runs:** add `-smoke`, e.g. `ext-03-unlearning-smoke`, so real and test outputs never share a dataset.
- **GroupNorm control arm** (`norm="gn"`): add `-gn`, e.g. `ext-03-unlearning-gn`.